In [ ]:
FEATURE_COLS = [
    'url_length',        # Total URL character count
    'num_dots',          # Number of dots (subdomains/extensions)
    'num_hyphens',       # Hyphens — common in fake domains
    'num_underscores',   # Underscores — unusual in legit URLs
    'num_slashes',       # Path depth
    'num_at',            # @ symbol hides real destination
    'num_question',      # Query parameters
    'num_equals',        # Parameter assignments
    'num_percent',       # URL encoding (may hide content)
    'num_digits',        # Digit count — auto-generated URLs
    'has_ip',            # IP instead of domain — strong indicator
    'has_https',         # Secure connection
    'has_suspicious_word', # login, verify, secure, etc.
    'num_subdomains',    # Excessive subdomains
    'hostname_length',   # Long hostname
    'path_length',       # Long path
    'double_slash',      # // in path — open redirect
    'has_at_in_url',     # @ in URL
    'num_suspicious_words', # Count of suspicious keywords
]
print(f"Total features: {len(FEATURE_COLS)}")
for i, f in enumerate(FEATURE_COLS, 1):
    print(f"  {i:2}. {f}")

Total features: 19
   1. url_length
   2. num_dots
   3. num_hyphens
   4. num_underscores
   5. num_slashes
   6. num_at
   7. num_question
   8. num_equals
   9. num_percent
  10. num_digits
  11. has_ip
  12. has_https
  13. has_suspicious_word
  14. num_subdomains
  15. hostname_length
  16. path_length
  17. double_slash
  18. has_at_in_url
  19. num_suspicious_words

In [ ]:
import sys
sys.path.insert(0, '../src')
from features.extract import extract_features

test_urls = [
    ('http://192.168.0.45/secure-login',          'Phishing — IP-based'),
    ('http://bankofegypt-login.evil.com/confirm',  'Phishing — Brand impersonation'),
    ('https://www.google.com',                     'Legitimate'),
]

for url, label in test_urls:
    f = extract_features(url)
    print(f"\n[{label}]")
    print(f"  URL         : {url}")
    print(f"  has_ip      : {f['has_ip']}")
    print(f"  has_https   : {f['has_https']}")
    print(f"  suspicious  : {f['has_suspicious_word']}")
    print(f"  subdomains  : {f['num_subdomains']}")
    print(f"  url_length  : {f['url_length']}")


[Phishing — IP-based]
  URL         : http://192.168.0.45/secure-login
  has_ip      : 1
  has_https   : 0
  suspicious  : 1
  subdomains  : 2
  url_length  : 34

[Phishing — Brand impersonation]
  URL         : http://bankofegypt-login.evil.com/confirm
  has_ip      : 0
  has_https   : 0
  suspicious  : 1
  subdomains  : 1
  url_length  : 46

[Legitimate]
  URL         : https://www.google.com
  has_ip      : 0
  has_https   : 1
  suspicious  : 0
  subdomains  : 1
  url_length  : 22

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../data/processed/phishtrace_features.csv')
corr = df[FEATURE_COLS + ['label']].corr()['label'].drop('label').sort_values(key=abs, ascending=False)

plt.figure(figsize=(10, 6))
colors = ['#e63946' if v > 0 else '#2ec4b6' for v in corr.values]
plt.barh(corr.index, corr.values, color=colors, alpha=0.8)
plt.title('Feature Correlation with Phishing Label', fontsize=13)
plt.xlabel('Pearson Correlation')
plt.axvline(0, color='gray', lw=0.8, linestyle='--')
plt.tight_layout()
plt.savefig('../reports/figures/feature_correlation.png', dpi=120)
plt.show()
print("Top positive correlators (→ phishing):")
print(corr[corr > 0].head(5).to_string())

Top positive correlators (→ phishing):
has_suspicious_word     0.421
num_suspicious_words    0.418
url_length              0.312
num_slashes             0.287
path_length             0.261